# Option Portfolio Metrics

Loads **your specific option positions** from `data/option_data.csv`, enriches them with
**fresh implied volatility** interpolated from the downloaded market chain (SQLite DB),
then computes Greeks, probabilities and payoff for those positions only.

The full downloaded chain is used purely as an IV surface reference — it is never directly analysed.

## 1 · Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

from skew import (
    compute_option_metrics,
    portfolio_greeks,
    market_implied_pdf,
    prob_profit_from_pdf,
    load_option_snapshots,
    fit_svi_slice,
    eval_svi_iv,
)

In [ ]:
# Auto-detect project root
_cwd = Path(os.getcwd())
if   (_cwd / "skew").exists():        _project_root = _cwd
elif (_cwd.parent / "skew").exists(): _project_root = _cwd.parent
else:                                  _project_root = _cwd

PORTFOLIO_CSV = str(_project_root / "data" / "option_data.csv")
DB_PATH       = str(_project_root / "data" / "options.db")

# None = use the most recent snapshot in the DB for IV interpolation
SNAPSHOT_DATE = None
DEFAULT_RF    = 0.04
CONTRACT_SIZE = 100     # shares per contract

print(f"Portfolio : {PORTFOLIO_CSV}")
print(f"Market DB : {DB_PATH}")

In [ ]:
## 2 · Load portfolio positions (minimal CSV)

# Only required columns: symbol, option_type, strike, expiry
# Optional: contracts (defaults to 1)
pf_raw = pd.read_csv(PORTFOLIO_CSV, usecols=lambda c: c in
    {"symbol", "option_type", "strike", "expiry", "contracts"})

pf = pf_raw.copy()

# Drop rows where required fields are missing
pf = pf.dropna(subset=["symbol", "option_type", "strike", "expiry"])

# Ensure string dtype (guards against mixed-type columns)
pf["symbol"] = pf["symbol"].astype(str).str.strip().str.upper()

# option_type → "C" / "P"
pf["option_type"] = (
    pf["option_type"].astype(str).str.upper().str.strip()
    .replace({"CALL": "C", "PUT": "P"})
)

# expiry as datetime (handles dd/mm/yyyy, yyyy-mm-dd, etc.)
pf["expiry"] = pd.to_datetime(pf["expiry"], dayfirst=True, errors="coerce")

# Drop rows where expiry couldn't be parsed
pf = pf.dropna(subset=["expiry"])

# T (years to expiry from today)
pf["T"] = ((pf["expiry"] - pd.Timestamp.today().normalize()).dt.days / 365.0).clip(lower=1e-6)

# contracts default
if "contracts" not in pf.columns:
    pf["contracts"] = 1

pf = pf.reset_index(drop=True)

print(f"Loaded {len(pf)} portfolio positions  |  tickers: {sorted(pf['symbol'].unique())}")
pf

In [ ]:
## 3 · Enrich each position from market chain (spot, IV, bid, ask, forward …)

def _load_chain(ticker, db_path, snapshot_date):
    raw = load_option_snapshots(ticker, db_path=db_path, snapshot_date=snapshot_date)
    if raw.empty:
        return pd.DataFrame()
    snap = snapshot_date or raw["snapshot_date"].max()
    df = raw[raw["snapshot_date"] == snap].copy()
    df["expiry"] = pd.to_datetime(df["expiry"])
    df["option_type"] = (
        df["option_type"].str.upper().str.strip()
        .replace({"CALL": "C", "PUT": "P"})
    )
    return df


def _svi_iv_for_strike(chain_exp, forward, T_exp, target_strike):
    """
    Fit SVI to OTM options in chain_exp and evaluate at target_strike.
    Falls back to np.interp on the raw slice if the fit fails.
    """
    F = forward
    calls_otm = chain_exp[(chain_exp["option_type"] == "C") & (chain_exp["strike"] >= F)]
    puts_otm  = chain_exp[(chain_exp["option_type"] == "P") & (chain_exp["strike"] <  F)]
    otm = pd.concat([puts_otm, calls_otm]).dropna(subset=["implied_vol"]).sort_values("strike")

    if len(otm) >= 5:
        params = fit_svi_slice(otm["strike"].values, otm["implied_vol"].values, F, T_exp)
        if params is not None:
            return float(eval_svi_iv(params, [target_strike], F, T_exp)[0])

    # Fallback: linear interpolation (no extrapolation beyond range)
    k_obs = chain_exp["strike"].values
    iv_obs = pd.to_numeric(chain_exp["implied_vol"], errors="coerce").values
    return float(np.interp(target_strike, k_obs, iv_obs))


def enrich_from_chain(ticker, option_type, expiry, strike, chain_cache):
    """Return a dict of market fields for a portfolio position."""
    empty = dict(spot=np.nan, implied_vol=np.nan, lastPrice=np.nan,
                 bid=np.nan, ask=np.nan, div_yield=0.0,
                 disc_factor=1.0, forward=np.nan)

    chain = chain_cache.get(ticker.upper(), pd.DataFrame())
    if chain.empty:
        return empty

    # closest expiry in chain
    exp_dt = pd.to_datetime(expiry)
    available = pd.to_datetime(chain["expiry"].unique())
    closest_exp = available[np.argmin(np.abs(available - exp_dt))]

    # all options at this expiry (for scalar fields)
    sl_all  = chain[chain["expiry"] == closest_exp].sort_values("strike")
    # same option type at this expiry (for bid/ask interp)
    sl_type = sl_all[sl_all["option_type"] == option_type.upper()]
    sl_ref  = sl_type if len(sl_type) >= 2 else sl_all

    if sl_all.empty:
        return empty

    spot      = float(sl_all["underlying_price"].iloc[0])
    div_yield = float(sl_all["div_yield"].iloc[0]) if "div_yield" in sl_all.columns else 0.0
    disc_fac  = float(sl_all["disc_factor"].iloc[0]) if "disc_factor" in sl_all.columns else 1.0
    forward   = float(sl_all["forward"].iloc[0]) if "forward" in sl_all.columns else spot
    T_exp     = float(sl_all["T"].iloc[0]) if "T" in sl_all.columns else max((closest_exp - pd.Timestamp.today()).days / 365.0, 1e-6)

    def _interp(sl, col):
        if col not in sl.columns:
            return np.nan
        vals = pd.to_numeric(sl[col], errors="coerce").values
        return float(np.interp(strike, sl["strike"].values, vals))

    bid = _interp(sl_ref, "bid")
    ask = _interp(sl_ref, "ask")
    mid = (0.5 * (bid + ask) if np.isfinite(bid) and np.isfinite(ask)
           else max(x for x in (bid, ask) if np.isfinite(x)) if any(np.isfinite(x) for x in (bid, ask))
           else np.nan)

    # ── SVI-fitted IV (interpolates + extrapolates convexly) ──────────────────
    iv = _svi_iv_for_strike(sl_all, forward, T_exp, strike)

    return dict(spot=spot, implied_vol=iv, lastPrice=mid,
                bid=bid, ask=ask, div_yield=div_yield,
                disc_factor=disc_fac, forward=forward)


# ── Build chain cache ─────────────────────────────────────────────────────────
tickers_needed = pf["symbol"].str.upper().unique()
chain_cache = {}
for tkr in tickers_needed:
    chain_cache[tkr] = _load_chain(tkr, DB_PATH, SNAPSHOT_DATE)
    n = len(chain_cache[tkr])
    snap = chain_cache[tkr]["snapshot_date"].max() if n else "N/A"
    print(f"  {tkr}: {n} chain rows  |  snapshot={snap}" if n
          else f"  {tkr}: NOT FOUND in DB — Greeks will use NaN IV")

# ── Enrich portfolio ──────────────────────────────────────────────────────────
enriched_rows = []
for _, row in pf.iterrows():
    market = enrich_from_chain(row["symbol"], row["option_type"],
                               row["expiry"], row["strike"], chain_cache)
    enriched_rows.append({**row.to_dict(), **market})

pf_enriched = pd.DataFrame(enriched_rows)
pf_enriched["r"] = DEFAULT_RF

n_iv = pf_enriched["implied_vol"].notna().sum()
print(f"\nMarket data enriched: {n_iv}/{len(pf_enriched)} positions have IV")
pf_enriched[["symbol","option_type","strike","expiry","spot","implied_vol","lastPrice","bid","ask","div_yield"]].round(4)

In [ ]:
## 4 · Compute Greeks & probabilities

metrics = compute_option_metrics(pf_enriched, default_rf=DEFAULT_RF, contract_size=CONTRACT_SIZE)

display_cols = [
    "symbol", "option_type", "strike", "expiry", "T", "implied_vol",
    "spot", "intrinsic", "extrinsic", "moneyness", "break_even", "max_loss",
    "prob_itm", "prob_profit", "prob_profit_ask",
    "delta", "gamma", "vega", "theta_day", "rho",
]
available = [c for c in display_cols if c in metrics.columns]
metrics[available].round(4)

In [ ]:
## 5 · Portfolio-level Greeks

port = portfolio_greeks(metrics)
pd.Series(port).rename("net_position").to_frame()

In [ ]:
## 6 · Probability chart — per position

labels = (
    metrics["symbol"] + " " + metrics["option_type"] +
    " K=" + metrics["strike"].astype(str) +
    " " + metrics["expiry"].dt.strftime("%b%y")
)

x = np.arange(len(metrics))
w = 0.28

fig, ax = plt.subplots(figsize=(max(10, len(metrics) * 1.4), 5))
ax.bar(x - w, metrics["prob_itm"],        width=w, label="P(ITM)",          color="steelblue")
ax.bar(x,     metrics["prob_profit"],      width=w, label="P(profit @ mid)", color="seagreen")
ax.bar(x + w, metrics["prob_profit_ask"],  width=w, label="P(profit @ ask)", color="orange")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylabel("Probability")
ax.set_title("Portfolio Positions — Probability Summary")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
## 7 · Portfolio payoff at expiry (all positions combined)

for exp in sorted(metrics["expiry"].dropna().unique()):
    legs = metrics[metrics["expiry"] == exp]
    if legs.empty:
        continue

    spot_val  = legs["spot"].iloc[0]
    S_range   = np.linspace(spot_val * 0.6, spot_val * 1.5, 500)
    total_pnl = np.zeros_like(S_range)

    for _, leg in legs.iterrows():
        K    = leg["strike"]
        prem = leg["lastPrice"] if pd.notna(leg.get("lastPrice")) else 0.0
        qty  = leg.get("contracts", 1) * CONTRACT_SIZE
        if leg["option_type"] == "C":
            total_pnl += qty * (np.maximum(S_range - K, 0) - prem)
        else:
            total_pnl += qty * (np.maximum(K - S_range, 0) - prem)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(S_range, total_pnl, lw=2, color="navy")
    ax.axhline(0, color="black", lw=0.8)
    ax.axvline(spot_val, color="grey", lw=1, linestyle="--", alpha=0.7, label=f"Spot {spot_val:.1f}")
    ax.fill_between(S_range, total_pnl, 0, where=(total_pnl > 0), alpha=0.2, color="green", label="Profit")
    ax.fill_between(S_range, total_pnl, 0, where=(total_pnl < 0), alpha=0.2, color="red",   label="Loss")
    for _, leg in legs.iterrows():
        ax.axvline(leg["strike"], color="purple", lw=0.8, linestyle=":", alpha=0.6)
    ax.set_xlabel("Underlying at expiry")
    ax.set_ylabel("P&L ($)")
    ax.set_title(f"Combined payoff — expiry {pd.Timestamp(exp).strftime('%d %b %Y')}")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
## 8 · Volatility smile — SVI fitted curve with portfolio strikes marked

for tkr in tickers_needed:
    chain = chain_cache.get(tkr, pd.DataFrame())
    if chain.empty:
        print(f"No market chain for {tkr} — skipping smile chart.")
        continue

    T_vals = chain["T"].dropna().unique() if "T" in chain.columns else []
    if len(T_vals) == 0:
        continue

    spot_val = float(chain["underlying_price"].iloc[0])
    fig, ax = plt.subplots(figsize=(12, 5))
    cmap = plt.cm.viridis(np.linspace(0, 1, min(len(T_vals), 10)))

    for t_val, color in zip(sorted(T_vals)[:10], cmap):
        sub = chain[np.isclose(chain["T"], t_val, atol=1e-3)]
        F = float(sub["forward"].iloc[0]) if "forward" in sub.columns else spot_val

        # OTM options for SVI fit (calls above F, puts below F)
        calls_otm = sub[(sub["option_type"] == "C") & (sub["strike"] >= F)]
        puts_otm  = sub[(sub["option_type"] == "P") & (sub["strike"] <  F)]
        otm = pd.concat([puts_otm, calls_otm]).dropna(subset=["implied_vol"]).sort_values("strike")

        label = f"T≈{t_val * 365:.0f}d"

        if len(otm) >= 5:
            params = fit_svi_slice(otm["strike"].values, otm["implied_vol"].values, F, t_val)
            if params is not None:
                # Plot SVI curve over a wide strike range (extrapolates smoothly)
                K_lo = min(otm["strike"].min(), spot_val * 0.60)
                K_hi = max(otm["strike"].max(), spot_val * 1.60)
                K_fine = np.linspace(K_lo, K_hi, 400)
                iv_fine = eval_svi_iv(params, K_fine, F, t_val) * 100
                ax.plot(K_fine, iv_fine, color=color, lw=1.8, alpha=0.8, label=label)
                # Raw liquid data as small dots
                ax.scatter(otm["strike"], otm["implied_vol"] * 100,
                           color=color, s=12, alpha=0.5, zorder=3)
                continue

        # Fallback: just plot raw data as a line
        if len(otm) > 1:
            ax.plot(otm["strike"], otm["implied_vol"] * 100,
                    color=color, lw=1.2, alpha=0.5, linestyle="--", label=label)

    # Mark portfolio strikes for this ticker
    pf_tkr = metrics[metrics["symbol"].str.upper() == tkr]
    y_top = ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 80
    for _, leg in pf_tkr.iterrows():
        ax.axvline(leg["strike"], color="red", lw=1.2, linestyle="--", alpha=0.8)
        ax.text(leg["strike"], y_top,
                f'{leg["option_type"]}K{leg["strike"]:.0f}',
                rotation=90, fontsize=7, color="red", va="top")

    ax.axvline(spot_val, color="black", lw=1.5, linestyle="--", alpha=0.5, label="Spot")
    ax.set_title(f"{tkr} — Implied Vol Smile  (SVI fit · dots = liquid strikes · red = portfolio)")
    ax.set_xlabel("Strike")
    ax.set_ylabel("IV (%)")
    ax.legend(fontsize=7, ncol=4, loc="upper right")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()